# 03_02 · Modelos — CNC Mill Tool Wear

LeaveOneOut CV con 18 muestras.


In [1]:
import pickle
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

with open('../data/processed/splits.pkl', 'rb') as f:
    splits = pickle.load(f)

X_cnc, y_cnc = splits['cnc_loo']

## 9. CNC Mill — LeaveOneOut (18 experimentos)

Con solo 18 muestras, un split 80/20 dejaría 3-4 muestras en test — insuficiente para
métricas estables. **LeaveOneOut CV** usa cada una de las 18 muestras como test, una a la vez.

El resultado (Accuracy, F1) promediado sobre 18 iteraciones es más representativo.

> **Contexto:** el dataset CNC es secundario en este proyecto — lo usamos para demostrar
> que el mismo enfoque funciona en un dominio de señales de sensores más rico.
> Los resultados más bajos (AUC~0.70) reflejan la dificultad del problema y el tamaño pequeño.


In [2]:
if 'cnc_loo' in splits:
    X_cnc, y_cnc = splits['cnc_loo']
    rf_cnc = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                    random_state=42, n_jobs=-1)
    loo = LeaveOneOut()
    y_pred_loo = cross_val_predict(rf_cnc, X_cnc, y_cnc, cv=loo)
    print('=== CNC Mill — Leave-One-Out ===')
    print(classification_report(y_cnc, y_pred_loo, target_names=['Unworn', 'Worn']))
else:
    print('CNC data not available — skipping.')

=== CNC Mill — Leave-One-Out ===
              precision    recall  f1-score   support

      Unworn       0.40      0.25      0.31         8
        Worn       0.54      0.70      0.61        10

    accuracy                           0.50        18
   macro avg       0.47      0.47      0.46        18
weighted avg       0.48      0.50      0.47        18



## 10. CNC — Comparativa LOO (más modelos)

Evaluamos 4 modelos con LOO-CV para identificar el mejor para CNC.

GradientBoosting suele ganar en datasets pequeños con muchas features porque
construye modelos de forma más controlada que RF (que puede sobreajustar con 187 features y 18 muestras).


In [3]:
if 'cnc_loo' in splits:
    from sklearn.ensemble import GradientBoostingClassifier
    
    cnc_models = {
        "LogReg":           LogisticRegression(class_weight="balanced", max_iter=1000),
        "RandomForest":     RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "SVM":              Pipeline([("sc", StandardScaler()), ("clf", SVC(class_weight="balanced", probability=True, random_state=42))]),
    }
    
    loo = LeaveOneOut()
    cnc_rows = []
    for name, model in cnc_models.items():
        y_pred = cross_val_predict(model, X_cnc, y_cnc, cv=loo)
        y_prob = cross_val_predict(model, X_cnc, y_cnc, cv=loo, method="predict_proba")[:,1]
        cnc_rows.append({
            "Modelo": name,
            "F1(worn)": f1_score(y_cnc, y_pred, pos_label=1),
            "ROC-AUC": roc_auc_score(y_cnc, y_prob),
        })
    
    cnc_comp = pd.DataFrame(cnc_rows).set_index("Modelo").round(4)
    print(cnc_comp.sort_values("ROC-AUC", ascending=False).to_string())
else:
    print('CNC data not available — skipping.')

                  F1(worn)  ROC-AUC
Modelo                             
GradientBoosting    0.7826   0.7000
LogReg              0.5833   0.5500
RandomForest        0.6087   0.4063
SVM                 0.6000   0.2250
